# Tutorial 9: Circuit Discovery

**Prerequisites:** T04 (Attention Heads), T08 (Activation Patching)

**What you'll learn:**
- What a "circuit" means in mechanistic interpretability
- How to systematically identify the minimal set of components responsible for a behavior
- How to use IRTK's circuit discovery tools
- How to evaluate circuit quality (faithfulness, completeness, minimality)

---

## What Is a Circuit?

A **circuit** is the minimal subset of a model's components (attention heads and MLP layers) that are responsible for a specific behavior. For example:

- The **induction circuit** (two attention heads that detect repeated patterns)
- The **IOI circuit** (about 26 heads in GPT-2 that handle indirect object identification)
- The **greater-than circuit** (heads that compare numbers)

Finding circuits is the central activity of mechanistic interpretability research.

### Properties of a Good Circuit

1. **Faithful**: The circuit reproduces the full model's behavior
2. **Complete**: All components that matter are included
3. **Minimal**: No unnecessary components are included

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=3, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

# Create a task: compare repeated vs non-repeated tokens
clean_tokens     = jnp.array([10, 20, 30, 20, 50])
corrupted_tokens = jnp.array([10, 20, 30, 99, 50])

logits_clean = model(clean_tokens)
logits_corrupt = model(corrupted_tokens)
target_token = int(jnp.argmax(logits_clean[-1]))

def metric(logits):
    return float(logits[-1, target_token])

clean_metric = metric(logits_clean)
corrupt_metric = metric(logits_corrupt)
print(f'Task: predicting token {target_token}')
print(f'Clean metric:     {clean_metric:.4f}')
print(f'Corrupted metric: {corrupt_metric:.4f}')
print(f'Effect size:      {clean_metric - corrupt_metric:.4f}')

## Step 1: Rank Components by Importance

The first step in circuit discovery is to rank all components by how much they matter for the behavior. We do this with activation patching.

In [ ]:
from irtk.patching import activation_patch

# Patch at every component
all_hooks = []
for l in range(cfg.n_layers):
    all_hooks.append(f'blocks.{l}.hook_attn_out')
    all_hooks.append(f'blocks.{l}.hook_mlp_out')

results = activation_patch(
    model, clean_tokens, corrupted_tokens,
    hook_names=all_hooks, metric_fn=metric,
)

# Sort by effect size (how much patching changes the metric)
effects = []
for name, patched_metric in results.items():
    effect = abs(patched_metric - corrupt_metric)
    effects.append((name, effect, patched_metric))

effects.sort(key=lambda x: -x[1])  # Sort by absolute effect

print('Components ranked by patching effect (denoising):')
for name, effect, patched in effects:
    short = name.replace('blocks.', 'L').replace('.hook_', ' ')
    bar = '#' * int(effect * 10)
    print(f'  {short:15s}: effect = {effect:.4f} {bar}')

## Step 2: Build the Circuit Greedily

Now we add components one at a time, starting from the most important, until the circuit is "faithful enough" (reproduces the full model's behavior above some threshold).

In [ ]:
# Greedy circuit building: keep adding the most important component
# Measure faithfulness by running the model with all OTHER components ablated

circuit = []
all_component_names = [name for name, _, _ in effects]

print('Building circuit greedily:\n')

for name, effect, _ in effects:
    circuit.append(name)
    
    # Ablate everything NOT in the circuit
    ablate_hooks = {}
    for comp_name in all_component_names:
        if comp_name not in circuit:
            ablate_hooks[comp_name] = lambda x, n: jnp.zeros_like(x)
    
    logits_circuit = model.run_with_hooks(clean_tokens, fwd_hooks=ablate_hooks)
    circuit_metric = metric(logits_circuit)
    
    # How close is this to the full model?
    total_effect = clean_metric - corrupt_metric
    if abs(total_effect) > 1e-10:
        faithfulness = (circuit_metric - corrupt_metric) / total_effect * 100
    else:
        faithfulness = 100.0
    
    short = name.replace('blocks.', 'L').replace('.hook_', ' ')
    print(f'  + {short:15s}: metric = {circuit_metric:+.4f}, '
          f'faithfulness = {faithfulness:.1f}%')
    
    if faithfulness > 90:
        print(f'\n  Circuit is {faithfulness:.1f}% faithful with {len(circuit)} components!')
        break

print(f'\nCircuit components: {[n.replace("blocks.", "L").replace(".hook_", " ") for n in circuit]}')

## Step 3: Use IRTK's Circuit Discovery Tools

IRTK provides automated circuit discovery via `experiments.find_circuit()`.

In [ ]:
from irtk.experiments import find_circuit

# Automatic circuit discovery
result = find_circuit(
    model, clean_tokens, corrupted_tokens,
    metric_fn=metric,
    threshold=0.8,  # 80% faithfulness target
)

print(f'Discovered circuit:')
print(f'  Components: {len(result["circuit"])}')
for comp in result['circuit']:
    short = comp.replace('blocks.', 'L').replace('.hook_', ' ')
    print(f'    {short}')
print(f'  Faithfulness: {result["faithfulness"]:.1%}')

## Step 4: Evaluate Circuit Quality

IRTK provides metrics for evaluating how good a circuit is.

In [ ]:
from irtk.circuit_evaluation import (
    circuit_faithfulness, circuit_completeness, circuit_minimality,
)

# Use the circuit we found
our_circuit = result['circuit']

# Faithfulness: does the circuit reproduce the full model's behavior?
faith = circuit_faithfulness(
    model, clean_tokens, our_circuit, metric_fn=metric,
)
print(f'Faithfulness: {faith["faithfulness"]:.1%}')
print(f'  (How well does the circuit match the full model?)')

# Completeness: is anything important missing?
comp = circuit_completeness(
    model, clean_tokens, corrupted_tokens,
    our_circuit, metric_fn=metric,
)
print(f'\nCompleteness: {comp["completeness"]:.1%}')
print(f'  (Are all important components included?)')

# Minimality: are there unnecessary components?
mini = circuit_minimality(
    model, clean_tokens, our_circuit, metric_fn=metric,
)
print(f'\nMinimality: {mini["minimality"]:.1%}')
print(f'  (Could we remove any component without losing faithfulness?)')
if mini.get('removable_components'):
    print(f'  Removable: {mini["removable_components"]}')

## Understanding Circuit Structure

Once we have a circuit, we can analyze how the components interact.

In [ ]:
# For each component in the circuit, what is its role?
print('Role analysis for circuit components:\n')

_, cache = model.run_with_cache(clean_tokens)
W_U = model.unembed.W_U

for comp_name in our_circuit:
    short = comp_name.replace('blocks.', 'L').replace('.hook_', ' ')
    act = cache[comp_name]  # [seq, d_model]
    
    # What does this component contribute to the target logit?
    logit_contrib = float(jnp.dot(act[-1], W_U[:, target_token]))
    
    # How much norm does it add?
    norm = float(jnp.linalg.norm(act[-1]))
    
    print(f'  {short:15s}:')
    print(f'    Logit contribution to token {target_token}: {logit_contrib:+.4f}')
    print(f'    Output norm: {norm:.4f}')
    print()

## Key Takeaways

1. **A circuit** is the minimal set of components responsible for a behavior
2. **Discovery method**: rank components by activation patching, add greedily until faithful
3. **Quality metrics**: faithfulness (does it work?), completeness (is anything missing?), minimality (is anything extra?)
4. **IRTK tools**: `find_circuit()` for discovery, `circuit_faithfulness/completeness/minimality()` for evaluation
5. **Circuit analysis** reveals how components interact to produce behaviors

**Next: [T10 — Sparse Autoencoders](T10_sparse_autoencoders.ipynb)** — Decomposing activations into interpretable features.